# Week 76, Day 2: HTTP

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w076d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
raw = (b"GET /courses/python?week=76 HTTP/1.1\r\n"
       b"Host: atlas.example.com\r\n"
       b"Accept: text/html\r\n"
       b"User-Agent: atlas-reader/1.0\r\n"
       b"\r\n")

head, _, body = raw.partition(b"\r\n\r\n")
start, *header_lines = head.decode("ascii").split("\r\n")
method, target, version = start.split(" ")
headers = dict(line.split(": ", 1) for line in header_lines)
print(method, target, version)
print(headers["Host"], len(body))

In [ ]:
from http import HTTPStatus

def response(status, body, content_type="text/plain; charset=utf-8"):
    data = body.encode("utf-8")
    head = f"HTTP/1.1 {status} {HTTPStatus(status).phrase}\r\nContent-Type: {content_type}\r\nContent-Length: {len(data)}\r\n\r\n"
    return head.encode("ascii") + data

print(response(200, "café"))
print(response(404, "nothing here").split(b"\r\n")[0])

chunked = b"4\r\nWiki\r\n6\r\npedia \r\nE\r\nin \r\n\r\nchunks.\r\n0\r\n\r\n"
pieces, rest = [], chunked
while True:
    size_line, _, rest = rest.partition(b"\r\n")
    size = int(size_line, 16)
    if size == 0:
        break
    pieces.append(rest[:size])
    rest = rest[size + 2:]
print(b"".join(pieces))

In [ ]:
from http.cookies import SimpleCookie

cookie = SimpleCookie()
cookie["session"] = "abc123"
cookie["session"]["max-age"] = 3600
cookie["session"]["path"] = "/"
cookie["session"]["secure"] = True
cookie["session"]["httponly"] = True
cookie["session"]["samesite"] = "Lax"
print(cookie.output())

incoming = SimpleCookie("session=abc123; theme=dark")
print({name: morsel.value for name, morsel in incoming.items()})

def conditional_get(etag, if_none_match):
    tags = [t.strip() for t in if_none_match.split(",")] if if_none_match else []
    return 304 if etag in tags or "*" in tags else 200

print(conditional_get('"v42"', '"v41", "v42"'), conditional_get('"v43"', '"v42"'))

> **Key idea.** HTTP/1.1 messages are a start line (request: method, target, version; response: version, status, reason), headers (case-insensitive, possibly repeated), a CRLF blank line and an optional body sized by Content-Length (in bytes) or chunked encoding (hex sizes, ending with 0). Host is required. GET, HEAD and OPTIONS are safe; those plus PUT and DELETE are idempotent and can be retried; POST and PATCH aren't. Status codes: 2xx success (200, 201, 204), 3xx redirects (301, 303, 307, 308) and 304 Not Modified, 4xx client errors (400, 401, 403, 404, 409, 422, 429), 5xx server errors. Content-Type and Accept describe bodies; Cache-Control, ETag and If-None-Match enable caching and 304s; Set-Cookie and Cookie carry state, protected by Secure, HttpOnly and SameSite; Location directs redirects.

### Exercise: Parsing a request

Write `parse_request(raw)`: parse an HTTP/1.1 request given as bytes into a dict with `"method"`, `"path"` (the target without its query), `"query"` (a dict from `parse_qs`), `"version"`, `"headers"` (a dict with **lower-case** names; repeated headers joined with `", "`) and `"body"` (exactly `Content-Length` bytes after the blank line, or `b""`). Lines end with `\r\n`. Raise `ValueError("bad request")` for a request line that isn't three parts, a version that doesn't start with `HTTP/`, a header line without `:`, a missing `Host`, or a body shorter than its `Content-Length`.

In [ ]:
from urllib.parse import urlsplit, parse_qs

def parse_request(raw):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parse():\n    \"start line, headers and body\"\n    raw = (b\"POST /api/orders?debug=1 HTTP/1.1\\r\\nHost: shop.example.com\\r\\nContent-Type: application/json\\r\\n\"\n           b\"Accept: text/html\\r\\naccept: application/json\\r\\nContent-Length: 8\\r\\n\\r\\n{\\\"q\\\": 1}\")\n    r = parse_request(raw)\n    assert (r[\"method\"], r[\"path\"], r[\"query\"], r[\"version\"]) == (\"POST\", \"/api/orders\", {\"debug\": [\"1\"]}, \"HTTP/1.1\")\n    assert r[\"headers\"][\"host\"] == \"shop.example.com\" and r[\"headers\"][\"accept\"] == \"text/html, application/json\"\n    assert r[\"body\"] == b'{\"q\": 1}'\n    assert parse_request(b\"GET / HTTP/1.1\\r\\nHost: a\\r\\n\\r\\n\")[\"body\"] == b\"\"\n    for bad in [b\"GET /\\r\\nHost: a\\r\\n\\r\\n\", b\"GET / FTP/1\\r\\nHost: a\\r\\n\\r\\n\", b\"GET / HTTP/1.1\\r\\nbroken\\r\\n\\r\\n\",\n                b\"GET / HTTP/1.1\\r\\nAccept: */*\\r\\n\\r\\n\", b\"POST / HTTP/1.1\\r\\nHost: a\\r\\nContent-Length: 10\\r\\n\\r\\nshort\"]:\n        try:\n            parse_request(bad)\n            assert False, bad\n        except ValueError as e:\n            assert str(e) == \"bad request\"")

### Exercise: Writing a response

Write `make_response(status, body=b"", headers=None)`: the bytes of an HTTP/1.1 response with the reason phrase from `HTTPStatus`, the given headers in order, and a `Content-Length` header (in bytes) added at the end unless the status is 204 or 304 (which have no body: raise `ValueError` if one is given). A `str` body is encoded as UTF-8, and a `Content-Type` of `text/plain; charset=utf-8` is added (before the length) unless the headers already have a content type (in any case). A `dict` or `list` body is encoded as compact JSON with `application/json`.

In [ ]:
import json
from http import HTTPStatus

def make_response(status, body=b"", headers=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_respond():\n    \"status lines, lengths and types\"\n    assert make_response(200, \"caf\u00e9\") == b\"HTTP/1.1 200 OK\\r\\nContent-Type: text/plain; charset=utf-8\\r\\nContent-Length: 5\\r\\n\\r\\ncaf\\xc3\\xa9\"\n    r = make_response(201, {\"id\": 7}, {\"Location\": \"/orders/7\"})\n    assert r == b'HTTP/1.1 201 Created\\r\\nLocation: /orders/7\\r\\nContent-Type: application/json\\r\\nContent-Length: 8\\r\\n\\r\\n{\"id\":7}'\n    assert make_response(204) == b\"HTTP/1.1 204 No Content\\r\\n\\r\\n\"\n    assert make_response(404, \"<h1>no</h1>\", {\"content-type\": \"text/html\"}).count(b\"ontent-\") == 2\n    try:\n        make_response(304, \"x\")\n        assert False\n    except ValueError:\n        pass")

### Exercise: Chunked bodies

Write `chunk(data, size)`: encode bytes with chunked transfer encoding, in chunks of at most `size` bytes (each as its length in lower-case hexadecimal, `\r\n`, the bytes, `\r\n`), ending with `0\r\n\r\n`. Then `dechunk(body)`: decode it, ignoring any **chunk extensions** after a `;` on a size line, and raising `ValueError` if a chunk is shorter than its size or isn't followed by `\r\n`, or if the final zero chunk is missing.

In [ ]:
def chunk(data, size):
    ___

def dechunk(body):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_chunks():\n    \"hexadecimal sizes\"\n    assert chunk(b\"Hello, chunked world\", 8) == b\"8\\r\\nHello, c\\r\\n8\\r\\nhunked w\\r\\n4\\r\\norld\\r\\n0\\r\\n\\r\\n\"\n    assert chunk(b\"\", 8) == b\"0\\r\\n\\r\\n\"\n    big = bytes(range(256)) * 3\n    assert dechunk(chunk(big, 100)) == big and b\"64\\r\\n\" in chunk(big, 100)\n    assert dechunk(b\"5;name=x\\r\\nhello\\r\\n0\\r\\n\\r\\n\") == b\"hello\"\n    for bad in [b\"5\\r\\nhell\\r\\n0\\r\\n\\r\\n\", b\"5\\r\\nhello\\r\\n\", b\"5\\r\\nhelloX0\\r\\n\\r\\n\"]:\n        try:\n            dechunk(bad)\n            assert False, bad\n        except ValueError:\n            pass")

### Exercise: Conditional requests

Write `answer(request_headers, etag, last_modified, max_age=60)`: decide the response to a `GET` for a resource whose current `etag` (a quoted string such as `'"v42"'`) and `last_modified` (a timezone-aware `datetime`) you know. `request_headers` has lower-case names. If there's an `if-none-match`, compare it (a comma-separated list, or `*`) with the etag, treating a weak tag `W/"v42"` as equal to `"v42"`, and ignore `if-modified-since`; otherwise, if there's an `if-modified-since` (an HTTP date, parsed with `email.utils.parsedate_to_datetime`), the resource is unchanged when `last_modified` isn't later than it. Return `(status, headers)`: `304` if unchanged, else `200`; headers always include `ETag`, `Last-Modified` (formatted with `email.utils.format_datetime(..., usegmt=True)`) and `Cache-Control: max-age=N`.

In [ ]:
from email.utils import parsedate_to_datetime, format_datetime

def answer(request_headers, etag, last_modified, max_age=60):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_conditional():\n    \"304 Not Modified\"\n    from datetime import datetime, timezone\n    lm = datetime(2026, 11, 1, 9, 30, tzinfo=timezone.utc)\n    status, headers = answer({}, '\"v42\"', lm)\n    assert status == 200 and headers == {\"ETag\": '\"v42\"', \"Last-Modified\": \"Sun, 01 Nov 2026 09:30:00 GMT\", \"Cache-Control\": \"max-age=60\"}\n    assert answer({\"if-none-match\": '\"v41\", W/\"v42\"'}, '\"v42\"', lm)[0] == 304\n    assert answer({\"if-none-match\": '\"v41\"'}, '\"v42\"', lm)[0] == 200 and answer({\"if-none-match\": \"*\"}, '\"v42\"', lm)[0] == 304\n    assert answer({\"if-modified-since\": \"Sun, 01 Nov 2026 09:30:00 GMT\"}, '\"v42\"', lm)[0] == 304\n    assert answer({\"if-modified-since\": \"Sat, 31 Oct 2026 09:30:00 GMT\"}, '\"v42\"', lm, max_age=5)[0] == 200\n    assert answer({\"if-none-match\": '\"old\"', \"if-modified-since\": \"Mon, 02 Nov 2026 00:00:00 GMT\"}, '\"v42\"', lm)[0] == 200")

### Exercise: Cookies

Write `set_cookie(name, value, max_age=None, path="/", secure=True, httponly=True, samesite="Lax")`: the value of a `Set-Cookie` header: `name=value`, then `Max-Age=N` if given, `Path=...`, `Secure`, `HttpOnly` and `SameSite=...` (each only if set), joined with `"; "`. Refuse (`ValueError`) a name that isn't letters, digits, `-` and `_`, a value containing `;`, `,`, spaces or quotes, and `SameSite=None` without `Secure` (browsers reject it). Then `read_cookies(header)`: a `Cookie` header's pairs as a dict (`"a=1; b=2"`), ignoring malformed parts without `=`.

In [ ]:
import re

def set_cookie(name, value, max_age=None, path="/", secure=True, httponly=True, samesite="Lax"):
    ___

def read_cookies(header):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cookies():\n    \"Set-Cookie and Cookie\"\n    assert set_cookie(\"session\", \"abc123\", max_age=3600) == \"session=abc123; Max-Age=3600; Path=/; Secure; HttpOnly; SameSite=Lax\"\n    assert set_cookie(\"theme\", \"dark\", secure=False, httponly=False, samesite=None, path=None) == \"theme=dark\"\n    for bad in [lambda: set_cookie(\"bad name\", \"x\"), lambda: set_cookie(\"a\", \"x;y\"), lambda: set_cookie(\"a\", \"x\", secure=False, samesite=\"None\")]:\n        try:\n            bad()\n            assert False\n        except ValueError:\n            pass\n    assert read_cookies(\"session=abc123; theme=dark; junk; x=\") == {\"session\": \"abc123\", \"theme\": \"dark\", \"x\": \"\"}")

### Exercise: What a method promises

Write `properties(method)`: a dict `{"safe": ..., "idempotent": ...}` for the methods in today's table (upper or lower case; unknown methods raise `ValueError`). Then `can_retry(method, status=None)`: whether a client may automatically resend a request that failed: `True` for an idempotent method when there was no response at all (`status` is `None`, a network error) or the status was `502`, `503` or `504`; `True` for any method on a `429` or `503` (the server said it didn't process it, so try later); otherwise `False`. Finally `kind(status)`: `"informational"`, `"success"`, `"redirect"`, `"client error"` or `"server error"`.

In [ ]:
SAFE = {"GET", "HEAD", "OPTIONS"}
IDEMPOTENT = SAFE | {"PUT", "DELETE"}
KNOWN = IDEMPOTENT | {"POST", "PATCH"}

def properties(method):
    ___

def can_retry(method, status=None):
    ___

def kind(status):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_semantics():\n    \"safe, idempotent and retryable\"\n    assert properties(\"get\") == {\"safe\": True, \"idempotent\": True} and properties(\"PUT\") == {\"safe\": False, \"idempotent\": True}\n    assert properties(\"POST\") == {\"safe\": False, \"idempotent\": False}\n    try:\n        properties(\"FETCH\")\n        assert False\n    except ValueError:\n        pass\n    assert can_retry(\"GET\") and can_retry(\"DELETE\", 502) and not can_retry(\"POST\") and not can_retry(\"POST\", 502)\n    assert can_retry(\"POST\", 429) and can_retry(\"PATCH\", 503) and not can_retry(\"GET\", 500) and not can_retry(\"PUT\", 404)\n    assert [kind(s) for s in (101, 204, 304, 404, 503)] == [\"informational\", \"success\", \"redirect\", \"client error\", \"server error\"]")

### Bug hunt: Lines that ended wrongly

The response's lines end with `\n` alone, but HTTP requires `\r\n`, so the strict parser can't find the end of the headers and this prints `no headers end`. Use `\r\n` throughout, so it prints `200 OK`.

In [ ]:
response = "HTTP/1.1 200 OK\nContent-Type: text/plain\nContent-Length: 2\n\nhi".encode()

head, sep, body = response.partition(b"\r\n\r\n")
if not sep:
    print("no headers end")
else:
    print(head.split(b"\r\n")[0].decode().split(" ", 1)[1])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"CRLF\"\n    expect_output(\"200 OK\")")

### Bug hunt: A length counted in characters

`Content-Length` must count **bytes**, but the code counts characters, so for `"Crème brûlée"` it's two short and the receiver cuts off the end: this prints `Crème brûl`. Count the encoded bytes, so it prints `Crème brûlée`.

In [ ]:
text = "Crème brûlée"
body = text.encode("utf-8")
length = len(text)

received = body[:length]
print(received.decode("utf-8", errors="ignore"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"bytes, not characters\"\n    expect_output(\"Cr\u00e8me br\u00fbl\u00e9e\")")

### Bug hunt: A header name with the wrong case

Header names are case-insensitive, and this server sent `content-type` in lower case, so looking up `"Content-Type"` in a plain dict finds nothing and this prints `None`. Store the names lower-cased and look them up lower-cased, so it prints `application/json`.

In [ ]:
lines = ["content-type: application/json", "content-length: 2"]
headers = {}
for line in lines:
    name, _, value = line.partition(":")
    headers[name.strip()] = value.strip()
print(headers.get("Content-Type"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"case-insensitive names\"\n    expect_output(\"application/json\")")

### Puzzle: An HTTP request

Arrange the lines of a request that creates an order with a JSON body. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the status code for "created", with a Location header
q = ___   # True or False: PUT is idempotent
r = ___   # the status for "your cached copy is still good"
s = ___   # len("é".encode("utf-8")), the Content-Length of that body
t = ___   # the cookie attribute that stops JavaScript reading a cookie

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 201 and q is True and r == 304, \"p, q and r\"\n    assert s == 2 and t == \"HttpOnly\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: reading responses from a stream

On a kept-alive connection, responses arrive one after another on the same byte stream, and the reader must know exactly where each ends. Write `read_responses(stream)`: parse every complete response from `stream` (bytes) and return `(responses, leftover)`: a list of dicts `{"status": int, "reason": str, "headers": dict (lower-case names), "body": bytes}`, and the bytes after the last complete response (an incomplete one waiting for more data). A body's length is given by `Content-Length`, or by `Transfer-Encoding: chunked` (decode it, using `dechunk` from the practice, given); responses with status `1xx`, `204` or `304` have no body. A response whose body is incomplete is left in `leftover`, along with everything after it.

In [ ]:
def dechunk_prefix(data):
    """Decode a chunked body at the start of data; return (body, bytes_used), or None if it's incomplete."""
    out, pos = b"", 0
    while True:
        end = data.find(b"\r\n", pos)
        if end < 0:
            return None
        size = int(data[pos:end].split(b";")[0].strip(), 16)
        pos = end + 2
        if size == 0:
            if data[pos:pos + 2] != b"\r\n":
                return None
            return out, pos + 2
        if len(data) < pos + size + 2:
            return None
        out += data[pos:pos + size]
        pos += size + 2

def read_responses(stream):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stream():\n    \"several responses on one connection\"\n    stream = (b\"HTTP/1.1 200 OK\\r\\nContent-Length: 5\\r\\n\\r\\nhello\"\n              b\"HTTP/1.1 304 Not Modified\\r\\nETag: \\\"v1\\\"\\r\\n\\r\\n\"\n              b\"HTTP/1.1 200 OK\\r\\nTransfer-Encoding: chunked\\r\\n\\r\\n3\\r\\nabc\\r\\n2\\r\\nde\\r\\n0\\r\\n\\r\\n\"\n              b\"HTTP/1.1 404 Not Found\\r\\nContent-Length: 9\\r\\n\\r\\nnot f\")\n    responses, leftover = read_responses(stream)\n    assert [(r[\"status\"], r[\"reason\"], r[\"body\"]) for r in responses] == [(200, \"OK\", b\"hello\"), (304, \"Not Modified\", b\"\"), (200, \"OK\", b\"abcde\")]\n    assert responses[1][\"headers\"] == {\"etag\": '\"v1\"'}\n    assert leftover == b\"HTTP/1.1 404 Not Found\\r\\nContent-Length: 9\\r\\n\\r\\nnot f\"\n    more, rest = read_responses(leftover + b\"ound\")\n    assert [(r[\"status\"], r[\"body\"]) for r in more] == [(404, b\"not found\")] and rest == b\"\"\n    assert read_responses(b\"HTTP/1.1 200 OK\\r\\nContent-Le\") == ([], b\"HTTP/1.1 200 OK\\r\\nContent-Le\")")

## Warm-up

That's a complete HTTP request, as it travels over the connection: a **request line** (method, target, version), **headers** (one per line, `Name: value`), and a blank line. Lines end with `\r\n` (carriage return and line feed). This one has no body. Everything a web framework gives you, from `request.path` to `request.headers`, starts by parsing exactly this.

## Messages

An HTTP/1.1 **request**:

```text
POST /api/orders HTTP/1.1
Host: shop.example.com
Content-Type: application/json
Content-Length: 27

{"product": 7, "qty": 2}
```

and its **response**:

```text
HTTP/1.1 201 Created
Content-Type: application/json
Content-Length: 31
Location: /api/orders/1042

{"id": 1042, "status": "new"}
```

- The **start line**: for a request, the **method**, the **target** (path and query) and the version; for a response, the version, a three-digit **status code**, and a **reason phrase** for humans.
- **Headers** follow, one per line. Names are **case-insensitive** (`content-type` and `Content-Type` are the same header); a header may appear more than once.
- An empty line ends the headers. Then the **body**, if there is one, whose length is given by `Content-Length` (in **bytes**) or by chunked encoding (below).
- **`Host`** is required in every HTTP/1.1 request: one server can host many sites on one address, and this says which one you want.

HTTP/2 and HTTP/3 send the same requests and responses (methods, paths, headers, status codes) in a compact binary framing, and can carry many at once over one connection; the meaning is unchanged, which is why the text form is the one to learn.

## Methods and status codes

**Methods** say what you want done to the resource named by the target:

| Method | Means | Safe | Idempotent |
|---|---|---|---|
| `GET` | read it | yes | yes |
| `HEAD` | read just the headers | yes | yes |
| `POST` | create something, or process this data | no | no |
| `PUT` | replace it with this | no | yes |
| `PATCH` | change part of it | no | no |
| `DELETE` | remove it | no | yes |
| `OPTIONS` | what may I do here? | yes | yes |

A **safe** method doesn't change anything on the server, so crawlers and prefetchers may send it freely. An **idempotent** method has the same effect whether it's sent once or many times (deleting order 7 twice leaves it deleted), so it can be **retried** after a network failure; a `POST` can't be safely retried without more care (Week 77).

**Status codes** are grouped by their first digit:

- **2xx success**: `200 OK`, `201 Created` (with a `Location` header), `204 No Content`.
- **3xx redirection**: `301 Moved Permanently`, `302 Found`, `303 See Other` (go and `GET` this other URL, after a form), `307` and `308` (redirect keeping the method), `304 Not Modified` (your cached copy is still good).
- **4xx client error** (the request was wrong): `400 Bad Request`, `401 Unauthorized` (you need to log in), `403 Forbidden` (you're logged in, but not allowed), `404 Not Found`, `405 Method Not Allowed`, `409 Conflict`, `422 Unprocessable Content` (valid syntax, invalid data), `429 Too Many Requests`.
- **5xx server error** (the server failed): `500 Internal Server Error`, `502 Bad Gateway`, `503 Service Unavailable`, `504 Gateway Timeout`.

`http.HTTPStatus` has them all: `HTTPStatus(404).phrase` is `"Not Found"`.

## Headers and bodies

The headers you'll meet constantly:

- **`Content-Type`** says what the body is: `text/html; charset=utf-8`, `application/json`, `application/x-www-form-urlencoded` (an HTML form), `multipart/form-data` (a form with files). **`Accept`** says what the client would like back (content negotiation).
- **`Content-Length`** is the body's size in **bytes**, not characters: `"café"` is 4 characters but 5 bytes in UTF-8.
- **`Transfer-Encoding: chunked`** is the alternative when the length isn't known in advance: the body is sent as chunks, each preceded by its size in **hexadecimal** on its own line, ending with a chunk of size 0.
- **`Connection: keep-alive`** (the HTTP/1.1 default) keeps the connection open for the next request; `Connection: close` ends it after this response.
- **`Authorization`** carries credentials, such as `Bearer TOKEN` (Week 77); **`User-Agent`** names the client.
- **`Content-Encoding: gzip`** says the body is compressed (with `Accept-Encoding` saying what the client can decompress).

## Caching, cookies and redirects

**Caching.** A response can say how long it may be reused: `Cache-Control: max-age=3600` (for an hour), `no-cache` (check with the server before reusing), `no-store` (never keep it: for private data), `private` (only in the user's browser, not in shared caches). To check cheaply, responses carry a **validator**: an **`ETag`** (an opaque version tag, such as a hash of the content) or `Last-Modified`. A client that has a copy sends a **conditional request**, `If-None-Match: "v42"`; if the resource hasn't changed, the server answers **`304 Not Modified`** with no body, saving the transfer.

**Cookies.** HTTP itself remembers nothing between requests. A server sets state in the browser with **`Set-Cookie: session=abc123; Max-Age=3600; Path=/; Secure; HttpOnly; SameSite=Lax`**, and the browser sends it back on later requests as **`Cookie: session=abc123`**. The attributes matter for security: `Secure` (only over HTTPS), `HttpOnly` (not readable by JavaScript, which protects session cookies from stolen scripts), `SameSite` (whether other sites' pages can make the browser send it; Week 79).

**Redirects.** A `3xx` response with a **`Location`** header tells the client to go elsewhere: a moved page (`301`, `308`), or, after a form is submitted, "see the result over here" (`303`), so that refreshing the page doesn't submit the form again. Clients should follow a limited number of redirects, and resolve a relative `Location` against the current URL (`urljoin`, yesterday).

## Practice

## On your own computer

`curl` shows HTTP as it really is. Try:

```bash
curl -v https://www.python.org/ -o /dev/null
```

```bash
curl -sI https://www.python.org/
```

`-v` prints the request and response headers (lines starting `>` and `<`), the TLS handshake, and the HTTP version used; `-I` sends a `HEAD` request and prints just the response headers. Look for `Cache-Control`, `ETag` and `Set-Cookie`, then send the ETag back with `curl -sI -H 'If-None-Match: "THE-ETAG"' URL` and watch for a `304`.

## Quiz

## Daily challenge

## Before you go

Open your browser's developer tools (the Network tab), load a page, and click one request: find its method, status, `Content-Type`, `Cache-Control` and cookies. Tomorrow: **calling APIs**: JSON over HTTP, clients and sessions, authentication, timeouts, retries and pagination.